# 语言模型零基础 04：OpenFst 组合、确定化与最小化

上一课编译并搜索了第一张图。这一课把多张图连接起来，并学习工程中最常用的图变换：

```text
arcsort → compose → project → rmepsilon → determinize → minimize → equivalent
```

完成本课后，你应该能够：

1. 用“关系连接”解释 composition；
2. 构造并组合一个极小 `L` 和 `G`；
3. 判断 input/output symbol table 是否兼容；
4. 解释投影、去 epsilon、确定化和最小化分别改变什么；
5. 使用 `fstequivalent` 验证优化前后语义没有改变。


In [1]:
from pathlib import Path
from math import log
import subprocess

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT = find_project_root()
LAB = ROOT / "openfst_lab" / "lesson04"
LAB.mkdir(parents=True, exist_ok=True)

def run_wsl(*args, check=True):
    result = subprocess.run(
        ["wsl", "-d", "Ubuntu", "--", *map(str, args)],
        text=True, capture_output=True, check=False, encoding="utf-8", errors="replace"
    )
    if check and result.returncode != 0:
        raise RuntimeError(f"命令失败：{args}\n{result.stderr}")
    return result

def to_wsl_path(path):
    resolved = Path(path).resolve()
    drive = resolved.drive.rstrip(":").lower()
    relative = resolved.relative_to(resolved.anchor).as_posix()
    return f"/mnt/{drive}/{relative}"

def write_lf(path, text):
    Path(path).write_text(text, encoding="utf-8", newline="\n")

def compile_fst(text_path, fst_path, input_symbols, output_symbols, acceptor=False):
    command = ["fstcompile"]
    if acceptor:
        command.append("--acceptor=true")
    command.extend([
        f"--isymbols={to_wsl_path(input_symbols)}",
        f"--osymbols={to_wsl_path(output_symbols)}",
        "--keep_isymbols=true",
        "--keep_osymbols=true",
        to_wsl_path(text_path),
        to_wsl_path(fst_path),
    ])
    run_wsl(*command)

def fst_info(path):
    return run_wsl("fstinfo", to_wsl_path(path)).stdout

def info_value(info, key):
    line = next(line for line in info.splitlines() if line.strip().startswith(key))
    return line.split()[-1]

print("实验目录：", LAB)
print("fstcompose：", run_wsl("which", "fstcompose").stdout.strip())


实验目录： <REPO_ROOT>\openfst_lab\lesson04


fstcompose： /usr/bin/fstcompose


## 1. Composition 是关系连接

如果：

```text
A: x → y
B: y → z
```

那么：

```text
A ∘ B: x → z
```

连接条件是 `A.output_label == B.input_label`。在 ASR 中，发音词典 `L` 把 phone/token 映射成 word，语言模型 `G` 在 word 空间中接受并打分，因此 `L ∘ G` 仍然接收 phone/token，输出受语言模型约束的 word。


## 2. 构造极小发音词典 L

词典包含：

```text
jin tian → jintian
tian qi  → tianqi
xin qing → xinqing
hen hao  → henhao
```

中间 phone 弧输出 `<eps>`，读完一个词时才输出 word。状态 0 同时是起点和终点，因此可以连续读取多个词。


In [2]:
phones_syms = LAB / "phones.syms"
words_syms = LAB / "words.syms"
l_text = LAB / "L.txt"
l_fst = LAB / "L.fst"

write_lf(phones_syms, "<eps> 0\njin 1\ntian 2\nqi 3\nxin 4\nqing 5\nhen 6\nhao 7\n")
write_lf(words_syms, "<eps> 0\njintian 1\ntianqi 2\nxinqing 3\nhenhao 4\n")
write_lf(l_text, """0 1 jin <eps> 0
1 0 tian jintian 0
0 2 tian <eps> 0
2 0 qi tianqi 0
0 3 xin <eps> 0
3 0 qing xinqing 0
0 4 hen <eps> 0
4 0 hao henhao 0
0
""")
compile_fst(l_text, l_fst, phones_syms, words_syms)
print(run_wsl(
    "fstprint",
    f"--isymbols={to_wsl_path(phones_syms)}",
    f"--osymbols={to_wsl_path(words_syms)}",
    to_wsl_path(l_fst),
).stdout)
print("L acceptor?", info_value(fst_info(l_fst), "acceptor"))


0	1	jin	<eps>
0	2	tian	<eps>
0	3	xin	<eps>
0	4	hen	<eps>
0
1	0	tian	jintian
2	0	qi	tianqi
3	0	qing	xinqing
4	0	hao	henhao

L acceptor? n


`L` 的 input symbol table 是 phone，output symbol table 是 word；两侧不同，所以它是 transducer，不是 acceptor。


## 3. 构造极小语言模型 G

`G` 接受两句话：

```text
jintian tianqi henhao    概率分支 0.7
jintian xinqing henhao   概率分支 0.3
```

两条竞争弧使用 `-log(probability)` 权重。


In [3]:
g_text = LAB / "G.txt"
g_fst = LAB / "G.fst"
p_tianqi = 0.7
p_xinqing = 0.3

def write_g(probability_tianqi):
    probability_xinqing = 1.0 - probability_tianqi
    write_lf(g_text, "\n".join([
        "0 1 jintian 0",
        f"1 2 tianqi {-log(probability_tianqi):.9f}",
        f"1 3 xinqing {-log(probability_xinqing):.9f}",
        "2 4 henhao 0",
        "3 4 henhao 0",
        "4",
        "",
    ]))

write_g(p_tianqi)
compile_fst(g_text, g_fst, words_syms, words_syms, acceptor=True)
print(run_wsl(
    "fstprint", f"--isymbols={to_wsl_path(words_syms)}",
    f"--osymbols={to_wsl_path(words_syms)}", to_wsl_path(g_fst)
).stdout)
print("G acceptor?", info_value(fst_info(g_fst), "acceptor"))
print("G input deterministic?", info_value(fst_info(g_fst), "input deterministic"))


0	1	jintian	jintian
1	2	tianqi	tianqi	0.356674939
1	3	xinqing	xinqing	1.20397282
2	4	henhao	henhao
3	4	henhao	henhao
4

G acceptor? y
G input deterministic? y


## 4. 先 arcsort，再 compose

组合需要匹配 `L.output` 与 `G.input`。在大图上，常见做法是：

- 按 output label 排序左图 `L`；
- 按 input label 排序右图 `G`；
- 再执行 composition。

排序通常是为了匹配器效率和可扩展性；不要把它误解成改变图所接受的语言。


In [4]:
l_sorted = LAB / "L.osorted.fst"
g_sorted = LAB / "G.isorted.fst"
lg_fst = LAB / "LG.fst"

run_wsl("fstarcsort", "--sort_type=olabel", to_wsl_path(l_fst), to_wsl_path(l_sorted))
run_wsl("fstarcsort", "--sort_type=ilabel", to_wsl_path(g_fst), to_wsl_path(g_sorted))
run_wsl("fstcompose", to_wsl_path(l_sorted), to_wsl_path(g_sorted), to_wsl_path(lg_fst))

lg_info = fst_info(lg_fst)
print("LG states：", info_value(lg_info, "# of states"))
print("LG arcs：", info_value(lg_info, "# of arcs"))
print("LG acceptor?", info_value(lg_info, "acceptor"))
print(run_wsl(
    "fstprint",
    f"--isymbols={to_wsl_path(phones_syms)}",
    f"--osymbols={to_wsl_path(words_syms)}",
    to_wsl_path(lg_fst),
).stdout)


LG states： 10
LG arcs： 10
LG acceptor? n
0	1	jin	<eps>
1	2	tian	jintian
2	3	tian	<eps>
2	4	xin	<eps>
3	5	qi	tianqi	0.356674939
4	6	qing	xinqing	1.20397282
5	7	hen	<eps>
6	8	hen	<eps>
7	9	hao	henhao
8	9	hao	henhao
9



观察 `LG`：输入仍是 phone，输出仍是 word，但只保留 `G` 允许的词序列，并继承 `G` 的语言模型权重。


## 5. 最常见失败：symbol table 不兼容

下面故意让同样的 word 字符串使用完全不同的整数 ID。OpenFst 比较的是标签 ID，并会利用嵌入图中的 symbol table 做兼容性检查。正确做法是让接口两侧从同一个 symbol table 生成，不是在报错后盲目删除 symbol table。


In [5]:
bad_words_syms = LAB / "bad_words.syms"
bad_g_fst = LAB / "G.bad_symbols.fst"
bad_lg = LAB / "LG.bad.fst"
write_lf(bad_words_syms, "<eps> 0\njintian 10\ntianqi 20\nxinqing 30\nhenhao 40\n")
compile_fst(g_text, bad_g_fst, bad_words_syms, bad_words_syms, acceptor=True)
failure = run_wsl(
    "fstcompose", to_wsl_path(l_sorted), to_wsl_path(bad_g_fst), to_wsl_path(bad_lg), check=False
)
print("return code：", failure.returncode)
print("stderr：", failure.stderr.strip())
assert failure.returncode != 0


return code： 1
stderr： WARNING: CompatSymbols: Symbol table checksums do not match. Table sizes are 5 and 5
FATAL: ComposeFst: Output symbol table of 1st argument does not match input symbol table of 2nd argument


排查空图或组合失败时依次检查：

1. `L.output symbols` 与 `G.input symbols` 是否相同；
2. `<eps>` 是否为 ID 0；
3. 两侧标签整数 ID 是否一致；
4. 是否真的存在可匹配路径；
5. `fstinfo` 中状态数、弧数和 error 属性；
6. 大图是否按匹配侧正确排序。


## 6. Project：只保留一侧标签

`LG` 是 phone→word transducer。输出投影后，原 output label 同时成为 input/output label，于是得到 word acceptor。投影会丢掉另一侧映射信息，不能从投影结果恢复原 phone 序列。


In [6]:
word_projected = LAB / "LG.output_projected.fst"
run_wsl(
    "fstproject", "--project_type=output", to_wsl_path(lg_fst), to_wsl_path(word_projected)
)
projected_info = fst_info(word_projected)
print("projected acceptor?", info_value(projected_info, "acceptor"))
print("projected epsilon arcs：", info_value(projected_info, "# of input/output epsilons"))
print(run_wsl(
    "fstprint", f"--isymbols={to_wsl_path(words_syms)}",
    f"--osymbols={to_wsl_path(words_syms)}", to_wsl_path(word_projected)
).stdout)


projected acceptor? y
projected epsilon arcs： 5


0	1	<eps>	<eps>
1	2	jintian	jintian
2	3	<eps>	<eps>
2	4	<eps>	<eps>
3	5	tianqi	tianqi	0.356674939
4	6	xinqing	xinqing	1.20397282
5	7	<eps>	<eps>
6	8	<eps>	<eps>
7	9	henhao	henhao
8	9	henhao	henhao
9



## 7. RmEpsilon：消除 epsilon 路径

投影后，词典中原来“不输出词”的弧变成 epsilon 弧。`fstrmepsilon` 会在当前 semiring 规则下合并 epsilon 路径和权重，保持加权语言等价。它不是简单删除含 `<eps>` 的文本行。


In [7]:
word_noeps = LAB / "LG.output.noeps.fst"
run_wsl("fstrmepsilon", to_wsl_path(word_projected), to_wsl_path(word_noeps))
before_eps = int(info_value(fst_info(word_projected), "# of input/output epsilons"))
after_eps = int(info_value(fst_info(word_noeps), "# of input/output epsilons"))
print("epsilon arcs before/after：", before_eps, "→", after_eps)
assert before_eps > 0 and after_eps == 0


epsilon arcs before/after： 5 → 0


## 8. Determinize 与 Minimize

- 确定化：同一状态对同一个 input label 最多有一条离开弧；
- 最小化：在保持加权语言等价的前提下减少可合并状态/弧；
- 不是每个任意 WFST 都可以直接确定化；加权 transducer 还涉及功能性、twins property 和权重半环等条件。

为清楚观察变化，下面专门构造一个非确定 acceptor：读入 `a` 后可以继续接受 `b` 或 `c`。


In [8]:
abc_syms = LAB / "abc.syms"
nfa_text = LAB / "nfa.txt"
nfa_fst = LAB / "nfa.fst"
det_fst = LAB / "det.fst"
min_fst = LAB / "min.fst"

write_lf(abc_syms, "<eps> 0\na 1\nb 2\nc 3\n")
write_lf(nfa_text, """0 1 a 0.1
0 2 a 0.4
1 3 b 0
2 3 c 0
3
""")
compile_fst(nfa_text, nfa_fst, abc_syms, abc_syms, acceptor=True)
print("NFA deterministic?", info_value(fst_info(nfa_fst), "input deterministic"))
run_wsl("fstdeterminize", to_wsl_path(nfa_fst), to_wsl_path(det_fst))
print("DET deterministic?", info_value(fst_info(det_fst), "input deterministic"))
run_wsl("fstminimize", to_wsl_path(det_fst), to_wsl_path(min_fst))

for name, path in [("NFA", nfa_fst), ("DET", det_fst), ("MIN", min_fst)]:
    info = fst_info(path)
    print(name, "states=", info_value(info, "# of states"),
          "arcs=", info_value(info, "# of arcs"),
          "deterministic=", info_value(info, "input deterministic"))


NFA deterministic? n


DET deterministic? y


NFA states= 4 arcs= 4 deterministic= n


DET states= 3 arcs= 3 deterministic= y
MIN states= 3 arcs= 3 deterministic= y


图变小或变得 deterministic 并不自动证明语义相同。优化流水线需要显式验证加权语言等价。

OpenFst 的精确 `fstequivalent` 要求两侧都是无 epsilon 的 deterministic acceptor。因此可以精确比较 `DET` 与 `MIN`；原始 NFA 与 DET 只能在这里使用 `--random=true` 做固定种子的多路径冒烟检查。随机检查不是数学证明，不能替代算法前置条件和正式验证。


In [9]:
random_nfa_det = run_wsl(
    "fstequivalent", "--random=true", "--npath=100", "--seed=42",
    to_wsl_path(nfa_fst), to_wsl_path(det_fst), check=False
)
exact_det_min = run_wsl(
    "fstequivalent", to_wsl_path(det_fst), to_wsl_path(min_fst), check=False
)
print("NFA ≈ DET 随机检查 return code：", random_nfa_det.returncode)
print("DET ≡ MIN 精确检查 return code：", exact_det_min.returncode)
print("说明：return code 0 表示该次等价性检查通过。")
assert random_nfa_det.returncode == 0
assert exact_det_min.returncode == 0


NFA ≈ DET 随机检查 return code： 0
DET ≡ MIN 精确检查 return code： 0
说明：return code 0 表示该次等价性检查通过。


## 9. 最短路径与语言模型概率滑块

改变 `G` 的两个分支概率，重新执行编译、排序、组合和最短路径。注意：`fstshortestpath` 可能重新编号状态，因此必须沿状态连接遍历输出，不能按 `fstprint` 的行顺序直接拼接 token。


In [10]:
import ipywidgets as widgets

best_lg = LAB / "LG.best.fst"

def ordered_path_labels(printed_text, label_column):
    arcs = {}
    destinations = set()
    finals = set()
    for line in printed_text.splitlines():
        fields = line.split()
        if len(fields) >= 4:
            source, destination = int(fields[0]), int(fields[1])
            arcs[source] = (destination, fields[label_column])
            destinations.add(destination)
        elif fields:
            finals.add(int(fields[0]))
    starts = set(arcs) - destinations
    if len(starts) != 1:
        raise ValueError(f"无法确定唯一初始状态：{starts}")
    state = starts.pop()
    labels, visited = [], set()
    while state not in finals:
        if state in visited or state not in arcs:
            raise ValueError("结果不是单条可完整遍历的路径")
        visited.add(state)
        state, label = arcs[state]
        if label != "<eps>":
            labels.append(label)
    return labels

def compose_and_best(probability_tianqi=0.7):
    write_g(probability_tianqi)
    compile_fst(g_text, g_fst, words_syms, words_syms, acceptor=True)
    run_wsl("fstarcsort", "--sort_type=ilabel", to_wsl_path(g_fst), to_wsl_path(g_sorted))
    run_wsl("fstcompose", to_wsl_path(l_sorted), to_wsl_path(g_sorted), to_wsl_path(lg_fst))
    run_wsl("fstshortestpath", to_wsl_path(lg_fst), to_wsl_path(best_lg))
    printed = run_wsl(
        "fstprint", f"--isymbols={to_wsl_path(phones_syms)}",
        f"--osymbols={to_wsl_path(words_syms)}", to_wsl_path(best_lg)
    ).stdout
    output_words = ordered_path_labels(printed, label_column=3)
    print(f"P(tianqi)={probability_tianqi:.2f}, P(xinqing)={1-probability_tianqi:.2f}")
    print("最佳输出：", " ".join(output_words))
    return output_words

widgets.interact(
    compose_and_best,
    probability_tianqi=widgets.FloatSlider(value=0.7, min=0.05, max=0.95, step=0.05),
);

default_words = compose_and_best(0.7)
assert default_words == ["jintian", "tianqi", "henhao"], default_words
print("默认组合最短路径校验通过。")


interactive(children=(FloatSlider(value=0.7, description='probability_tianqi', max=0.95, min=0.05, step=0.05),…

P(tianqi)=0.70, P(xinqing)=0.30
最佳输出： jintian tianqi henhao
默认组合最短路径校验通过。


## 10. 操作速查表

| 命令 | 主要作用 | 应保持什么 | 常见前置检查 |
|---|---|---|---|
| `fstarcsort` | 按输入或输出标签排列弧 | 加权关系 | 排序侧是否正确 |
| `fstcompose` | 连接左图输出与右图输入 | 关系组合语义 | 中间标签空间、symbol table |
| `fstproject` | 只保留输入侧或输出侧 | 被选一侧的加权语言 | 是否愿意丢弃另一侧映射 |
| `fstrmepsilon` | 合并 epsilon 路径 | 加权关系 | semiring、epsilon 环 |
| `fstdeterminize` | 消除同输入标签竞争弧 | 加权语言 | 是否可确定化 |
| `fstminimize` | 合并等价状态 | 加权语言 | 通常先确定化 |
| `fstequivalent` | 检查两个 acceptor 的加权语言 | — | 精确模式要求无 epsilon 且 deterministic；另有随机模式 |


## 11. 自动判题


In [11]:
# 请修改七个答案
answer_1 = ""  # A∘B 匹配 A 的 output 还是 input 与 B.input？
answer_2 = ""  # 左图为了 compose 常按哪侧排序？填 input/output
answer_3 = ""  # 输出投影后得到 input/output labels 相同的 acceptor 还是 transducer？
answer_4 = ""  # 消除 epsilon 路径的命令
answer_5 = ""  # 确定图中，同一状态同一 input 最多有几条离开弧？填数字
answer_6 = ""  # 检查加权 acceptor 语义等价的命令
answer_7 = ""  # L.output 应与 G 的 input 还是 output labels 匹配？

checks = [
    str(answer_1).strip().lower() == "output",
    str(answer_2).strip().lower() == "output",
    str(answer_3).strip().lower() == "acceptor",
    str(answer_4).strip().lower() == "fstrmepsilon",
    str(answer_5).strip() == "1",
    str(answer_6).strip().lower() == "fstequivalent",
    str(answer_7).strip().lower() == "input",
]
for number, ok in enumerate(checks, 1):
    print(("✅" if ok else "❌"), f"第 {number} 题")
print(f"得分：{sum(checks)}/7")
if all(checks):
    print("通过：你已经掌握 OpenFst 核心图操作的第一轮实操。")
else:
    print("回到对应命令，先用 fstinfo/fstprint 找证据，再修改答案。")


❌ 第 1 题
❌ 第 2 题
❌ 第 3 题
❌ 第 4 题
❌ 第 5 题
❌ 第 6 题
❌ 第 7 题
得分：0/7
回到对应命令，先用 fstinfo/fstprint 找证据，再修改答案。


<details><summary>完成后展开参考答案</summary>

1. `output`；2. `output`；3. `acceptor`；4. `fstrmepsilon`；5. `1`；6. `fstequivalent`；7. `input`。

</details>

## 离场票

不看上文，画出 `phone --L→ word --G→ scored word`，写出 `L∘G` 的输入、输出和权重来自哪里。然后解释：为什么状态数减少不能证明优化正确，为什么还要运行 `fstequivalent`。

下一课：打开 `语言模型零基础_05_从语料到ARPA与Gfst.ipynb`，使用 KenLM 训练真实 N-gram，读懂 ARPA，并把它转换成 OpenFst `G.fst`。
